In [1]:
# 步骤 1：下载 CB513 数据集（Zhou & Troyanskaya, ICML 2014 —— 二级结构预测标准基准）
# 来源：Zenodo record 7764556（数据集作者维护的官方镜像，原 Princeton 链接已失效）
# 注意：该文件名为 .gz 但实际是未压缩的 .npy（md5 与 Zenodo 官方记录一致）
mkdir -p /workspace/data && cd /workspace/data
if [ ! -f cb513+profile_split1.npy ]; then
  curl -sL --retry 3 -o cb513_raw.bin "https://zenodo.org/records/7764556/files/cb513%2Bprofile_split1.npy.gz?download=1"
  echo "a6ca35acd2a453b0186e3fe4af23011f  cb513_raw.bin" | md5sum -c -
  mv cb513_raw.bin cb513+profile_split1.npy
fi
ls -la /workspace/data

total 160225


drwxr-xr-x. 2 root root        38 Nov 10 11:36 .


drwxrwxrwx. 8 root root        80 Nov 10 11:34 ..


-rw-r--r--. 1 root root 164068880 Nov 10 11:36 cb513+profile_split1.npy


In [2]:
# 步骤 2：解析数据 —— 每条蛋白是一个 (700 残基 × 57 特征) 的矩阵，padding 到 700
# 57 维特征布局（官方 readme）：
#   [0,22)   氨基酸 one-hot（20 标准残基 + X + NoSeq），顺序 A,C,E,D,G,F,I,H,K,M,L,N,Q,P,S,R,T,W,V,Y,X
#   [22,31)  Q8 二级结构标签（8 态 + NoSeq），顺序 L,B,E,G,I,H,S,T
#   [31,33)  N/C 末端标记
#   [33,35)  溶剂可及度（本教程不用）
#   [35,57)  序列谱 profile（PSI-BLAST，22 维）
import numpy as np

data = np.load('/workspace/data/cb513+profile_split1.npy').reshape(514, 700, 57).astype(np.float32)

aa   = data[:, :, 0:22]
q8   = data[:, :, 22:31]
prof = data[:, :, 35:57]

# mask：NoSeq=0 的位置是真实残基
is_real = aa[:, :, 21] == 0
lengths = is_real.sum(axis=1).astype(np.int64)

# 模型输入特征：21 维氨基酸 one-hot + 22 维序列谱 = 43 维
X = np.concatenate([aa[:, :, :21], prof], axis=-1)          # (514, 700, 43)

# Q8 → Q3 映射（领域标准做法）：H,G,I→H；E,B→E；S,T,L→C
# Q8 索引顺序: 0=L, 1=B, 2=E, 3=G, 4=I, 5=H, 6=S, 7=T, 8=NoSeq
# 查找表共 9 项：NoSeq（padding 位置）直接映射为 -100（损失中忽略）
q3_lut = np.array([2, 1, 1, 0, 0, 0, 2, 2, -100])   # 0=H(螺旋), 1=E(折叠), 2=C(卷曲)
Y = q3_lut[q8.argmax(axis=-1)].astype(np.int64)             # (514, 700)

# ---- 数据检查 ----
print('蛋白条数: %d | 最长/平均序列长度: %d / %.1f' % (len(X), lengths.max(), lengths.mean()))
assert (aa[is_real][:, :21].sum(axis=1) == 1).all(), 'one-hot 校验失败'
assert (Y[~is_real] == -100).all() and (Y[is_real] >= 0).all(), 'mask 校验失败'

names = ['H(螺旋)', 'E(折叠)', 'C(卷曲)']
cnts = [(Y[is_real] == k).sum() for k in range(3)]
for n, c in zip(names, cnts):
    print('%s: %d 残基 (%.1f%%)' % (n, c, 100 * c / sum(cnts)))
print('总残基数:', sum(cnts))

/tmp/ipykernel_421/1897524055.py:10: UserWarning: Reading `.npy` or `.npz` file required additional header parsing as it was created on Python 2. Save the file again to speed up loading and avoid this warning.
  data = np.load('/workspace/data/cb513+profile_split1.npy').reshape(514, 700, 57).astype(np.float32)


蛋白条数: 514 | 最长/平均序列长度: 700 / 164.9
H(螺旋): 29319 残基 (34.6%)
E(折叠): 19197 残基 (22.6%)
C(卷曲): 36249 残基 (42.8%)
总残基数: 84765


In [5]:
# 步骤 3：数据管线 —— 划分数据集 + Dataset/DataLoader + 变长序列处理
# 要点：每条蛋白长度不同（20~700 残基），用 pack_padded_sequence 让 RNN 只处理真实残基
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

torch.manual_seed(42); np.random.seed(42)

# 划分：380 训练 / 50 验证 / 84 测试（固定随机种子，可复现）
perm = np.random.default_rng(42).permutation(514)
splits = {'train': perm[:380], 'val': perm[380:430], 'test': perm[430:]}

class ProteinSS(Dataset):
    """逐残基分类数据集：返回单条蛋白的 (序列特征, 逐残基标签)，均只取真实残基部分"""
    def __init__(self, X, Y, lengths, idx):
        self.X, self.Y, self.L, self.idx = X, Y, lengths, idx
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]; L = self.L[j]
        return (torch.from_numpy(np.ascontiguousarray(self.X[j, :L])),   # (L, 43)
                torch.from_numpy(np.ascontiguousarray(self.Y[j, :L])))   # (L,)  ← 标签同样截断

def collate(batch):
    """把不同长度的蛋白拼成一个 batch：短序列补齐，标签补 -100（损失中忽略）"""
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs])
    X = pad_sequence(xs, batch_first=True)                       # (B, Lmax, 43)
    Y = pad_sequence(ys, batch_first=True, padding_value=-100)   # (B, Lmax)
    return X, Y, lengths

loaders = {k: DataLoader(ProteinSS(X, Y, lengths, idx), batch_size=32,
                         shuffle=(k == 'train'), collate_fn=collate)
           for k, idx in splits.items()}

xb, yb, lb = next(iter(loaders['train']))
print('一个 batch: 特征 %s, 标签 %s, 长度 %s' % (tuple(xb.shape), tuple(yb.shape), lb[:5].tolist()))
assert xb.shape[1] == yb.shape[1], '特征与标签长度不一致'

一个 batch: 特征 (32, 700, 43), 标签 (32, 700), 长度 [68, 129, 171, 195, 86]


In [4]:
# 步骤 4：模型 —— 2 层双向 LSTM + 全连接头，逐残基三分类
# 要点：
#   - bidirectional=True：正向读一遍、反向读一遍，拼接每个残基的上下文（文本任务常用；
#     注意：若是“预测未来”的时间序列任务则不能用双向，会泄露未来信息）
#   - pack_padded_sequence：RNN 不在 padding 位置上浪费计算、不污染隐状态
#   - 输出层作用在每个时间步上（多对多模式），每个残基独立分类
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class BiLSTMSS(nn.Module):
    def __init__(self, input_size=43, hidden_size=64, num_layers=2,
                 num_classes=3, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * hidden_size, num_classes)   # 双向 → 2*hidden

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)                          # 只含真实残基位置
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        return self.fc(self.dropout(out))                   # (B, L, 3) 每个残基一个三分类 logits

model = BiLSTMSS()
n_params = sum(p.numel() for p in model.parameters())
print(model)
print('\n可训练参数量: %d' % n_params)

BiLSTMSS(
  (lstm): LSTM(43, 64, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=128, out_features=3, bias=True)
)

可训练参数量: 155523


In [6]:
# 步骤 5：训练 —— 完整训练循环（这是 LSTM 使用流程的核心模板）
# 要点：
#   - CrossEntropyLoss(ignore_index=-100)：padding 位置不参与损失
#   - clip_grad_norm_：RNN/LSTM 防梯度爆炸的标配
#   - 每轮在验证集上评估，保留验证准确率最高的模型（早停思想）
import time

crit = nn.CrossEntropyLoss(ignore_index=-100)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_acc, best_state = 0.0, None

def run_epoch(loader, train=False):
    model.train() if train else model.eval()
    tot_loss, tot_res, correct = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for X, Yb, L in loader:
            logits = model(X, L)
            loss = crit(logits.reshape(-1, 3), Yb.reshape(-1))
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
            m = Yb != -100
            tot_loss += loss.item() * m.sum().item()   # 按残基加权
            correct += (logits.argmax(-1)[m] == Yb[m]).sum().item()
            tot_res += m.sum().item()
    return tot_loss / tot_res, correct / tot_res

t0 = time.time()
for epoch in range(1, 16):
    tr_loss, _ = run_epoch(loaders['train'], train=True)
    va_loss, va_acc = run_epoch(loaders['val'], train=False)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)
    if va_acc > best_acc:
        best_acc = va_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    print('epoch %2d | train loss %.4f | val loss %.4f | val Q3 acc %.3f'
          % (epoch, tr_loss, va_loss, va_acc))

model.load_state_dict(best_state)
print('\n训练完成，用时 %.1f 分钟 | 最佳验证 Q3 准确率 %.3f' % ((time.time() - t0) / 60, best_acc))

epoch  1 | train loss 1.0658 | val loss 1.0569 | val Q3 acc 0.435


epoch  2 | train loss 1.0257 | val loss 0.9774 | val Q3 acc 0.570


epoch  3 | train loss 0.9168 | val loss 0.8677 | val Q3 acc 0.592


epoch  4 | train loss 0.8217 | val loss 0.7687 | val Q3 acc 0.675


epoch  5 | train loss 0.7554 | val loss 0.7281 | val Q3 acc 0.690


epoch  6 | train loss 0.7151 | val loss 0.6968 | val Q3 acc 0.708


... (5 outputs left out)

epoch 12 | train loss 0.6137 | val loss 0.6301 | val Q3 acc 0.744


epoch 13 | train loss 0.6111 | val loss 0.6099 | val Q3 acc 0.747


epoch 14 | train loss 0.6158 | val loss 0.6125 | val Q3 acc 0.747


epoch 15 | train loss 0.6064 | val loss 0.6088 | val Q3 acc 0.745

训练完成，用时 4.6 分钟 | 最佳验证 Q3 准确率 0.747


In [7]:
# 步骤 6a：训练曲线 —— 左：损失；右：验证集 Q3 准确率
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'

import os
outdir = '/mnt/results/ss_lstm_tutorial'
os.makedirs(outdir, exist_ok=True)

epochs = np.arange(1, 16)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].plot(epochs, history['train_loss'], 'o-', color='#0279EE', label='Train')
axes[0].plot(epochs, history['val_loss'], 's-', color='#FF9400', label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Loss'); axes[0].legend(frameon=False)
axes[1].plot(epochs, history['val_acc'], 's-', color='#75A025')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Q3 accuracy')
axes[1].set_title('Validation accuracy')
for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(f'{outdir}/training_curve.svg'); fig.savefig(f'{outdir}/training_curve.png', dpi=200)
plt.show()
print('图已保存:', outdir)

图已保存: /mnt/results/ss_lstm_tutorial


In [8]:
# 步骤 6b：测试集评估 —— 总体 Q3 准确率 + 各类 P/R/F1 + 混淆矩阵
# 指标均从混淆矩阵手工计算（透明可查，不依赖 sklearn）
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for X, Yb, L in loaders['test']:
        logits = model(X, L)
        m = Yb != -100
        all_pred.append(logits.argmax(-1)[m].numpy())
        all_true.append(Yb[m].numpy())
all_pred = np.concatenate(all_pred); all_true = np.concatenate(all_true)

names = ['H (helix)', 'E (strand)', 'C (coil)']
K = 3
cm = np.zeros((K, K), dtype=int)
for t, p in zip(all_true, all_pred):
    cm[t, p] += 1

q3_acc = np.trace(cm) / cm.sum()
print('测试集: %d 个残基 | 总体 Q3 准确率: %.3f' % (cm.sum(), q3_acc))
print('\n类\t精确率\t召回率\tF1')
for k in range(K):
    tp = cm[k, k]
    prec = tp / cm[:, k].sum()
    rec = tp / cm[k, :].sum()
    f1 = 2 * prec * rec / (prec + rec)
    print('%s\t%.3f\t%.3f\t%.3f' % (names[k], prec, rec, f1))

fig, ax = plt.subplots(figsize=(4.6, 4))
norm = cm / cm.sum(axis=1, keepdims=True)
im = ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
for i in range(K):
    for j in range(K):
        ax.text(j, i, '%d\n(%.0f%%)' % (cm[i, j], 100 * norm[i, j]),
                ha='center', va='center', fontsize=10,
                color='white' if norm[i, j] > 0.5 else 'black')
ax.set_xticks(range(K), names); ax.set_yticks(range(K), names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion matrix (residues)')
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(f'{outdir}/confusion_matrix.svg'); fig.savefig(f'{outdir}/confusion_matrix.png', dpi=200)
plt.show()
print('\n参考基线：Heffernan et al. 2017 双向 LSTM（更复杂特征+模型）Q3 ≈ 84%；本教程精简版（380 条训练蛋白）约 75% 属正常水平')

测试集: 13393 个残基 | 总体 Q3 准确率: 0.737



类	精确率	召回率	F1
H (helix)	0.773	0.796	0.784
E (strand)	0.666	0.669	0.668
C (coil)	0.745	0.728	0.736



参考基线：Heffernan et al. 2017 双向 LSTM（更复杂特征+模型）Q3 ≈ 84%；本教程精简版（380 条训练蛋白）约 75% 属正常水平


# LSTM 使用流程五要点（本 notebook 的模板价值）

1. **编码**：把每个时间步变成定长向量 —— 蛋白残基 = 21 维 one-hot + 22 维序列谱；文本 = 词 ID → Embedding；时间序列 = 归一化后的特征。输入永远是 `(batch, seq_len, input_size)`。
2. **变长处理**：`Dataset` 只取真实长度，`collate` 里 `pad_sequence` 补齐 + `pack_padded_sequence` 让 RNN 跳过 padding。
3. **模型**：`nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)` + 作用在**每个时间步**上的输出头（多对多）；预测未来/因果任务则去掉 bidirectional、只取最后一步（多对一）。
4. **训练循环**：`loss.backward()` 就是 BPTT；`clip_grad_norm_` 防爆炸；`ignore_index` 让 padding 不进损失；每轮在验证集上选最优模型。
5. **评估**：逐残基任务看总体准确率 + 每类 P/R/F1 + 混淆矩阵（这里 E 类最难，与文献一致）。

## 升级到标准基准协议

- 用 2 GB 的 `cullpdb+profile_6133.npy.gz`（Zenodo 同一 record）训练，CB513 只做测试 —— 这是文献标准协议，预期 Q3 可达 ~80%+
- 改进方向：加入溶剂可及度特征、加深加宽、学习率调度、用 Q8 八态任务
- 文献参考：Heffernan et al. 2017 (Bioinformatics) BiLSTM Q3 ≈ 84%

In [9]:
ls -la /mnt/results/ss_lstm_tutorial/

ls: /mnt/results/ss_lstm_tutorial/confusion_matrix.png


ls: /mnt/results/ss_lstm_tutorial/confusion_matrix.svg


ls: /mnt/results/ss_lstm_tutorial/training_curve.png


ls: /mnt/results/ss_lstm_tutorial/training_curve.svg


total 200


drwxrwxrwx. 2 ubuntu root     0 Nov 10 11:29 .


drwxrwxrwx. 2 ubuntu root     0 Nov 10 11:29 ..


-rw-rw-rw-  1 ubuntu root 77472 Nov 10 11:42 confusion_matrix.png


-rw-rw-rw-  1 ubuntu root 14397 Nov 10 11:42 confusion_matrix.svg


-rw-rw-rw-  1 ubuntu root 87524 Nov 10 11:42 training_curve.png


-rw-rw-rw-  1 ubuntu root 23956 Nov 10 11:42 training_curve.svg
